In [0]:
%run "../00. Common/Config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"


In [0]:
sprints_df = spark.table(bronze_table)

In [0]:
display(sprints_df)

In [0]:
sprints_selected_df = sprints_df.drop("url")

In [0]:
sprints_renamed_df = (
    sprints_selected_df
        .withColumnsRenamed({
            "raceName": "race_name",
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "position": "final_position",
            "number": "car_number",
            "positionText": "final_position_text",
        })
)

In [0]:
import pyspark.sql.functions as F
sprints_valid_df = sprints_renamed_df.filter(
    F.col("season").isNotNull() &
    F.col("round").isNotNull() &
    F.col("constructor_id").isNotNull() &
    F.col("driver_id").isNotNull()
)

In [0]:
sprints_distinct_df = sprints_valid_df.dropDuplicates(["season", "round", "constructor_id", "driver_id"])

In [0]:
display(sprints_distinct_df)

In [0]:
import pyspark.sql.functions as F
sprints_final_df = (
    sprints_distinct_df
        .withColumn('race_name',F.initcap("race_name"))
)

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))